In [51]:
# Base de dados
# 1: Spam, 0: Não Spam

emails = [
    "Free money now click here",
    "Call me back later please",
    "Win a free cruise vacation",
    "Are we meeting for lunch",
    "Click here to clain your free reward"
]
values = [1, 0, 1, 0, 1] 
labels = ['spam', 'n_spam']

In [52]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)

In [53]:
 # Montando a base de dados e criando vetores soma

database = list(zip(X.toarray(), values))

emails_not_spam = list(filter(lambda x: x[1] == 0, database))
emails_spam = list(filter(lambda x: x[1] == 1, database))

soma_not_spam = sum(arr for arr, _ in emails_not_spam)
soma_spam = sum(arr for arr, _ in emails_spam)

soma_emails_dict = {
    0:soma_not_spam,
    1:soma_spam
}

In [54]:
def py(classe):
    # Calcula a probabilidade a priori da classe

    total_emails = len(values)
    total_spam = values.count(1)
    total_n_spam = values.count(0)

    if classe == 1:
        y = total_spam / total_emails
    elif classe == 0:
        y = total_n_spam / total_emails
    else:
        print('Erro. Insira a classe')
        return False

    return y

def calcular_frequencia_palavra(aparicoes_da_palavra, total_termos_classe, tamanho_vocabulario):
    # 1º argumento: Quantas vezes a palavra atual apareceu na classe no treino
    # 2º argumento: A soma de todas as palavras que foram escritas nessa classe no treino
    # 3º argumento: tamanho do corpus para Suavização de Laplace

    prob_palavra = (aparicoes_da_palavra + 1) / (total_termos_classe + tamanho_vocabulario)

    return prob_palavra

def p_cond(vetor, classe):
    # Calcula da verossimilhança (Likelihood: p(x | y))
    # Faz o produtório das probabilidades de cada palavra existir naquela classe

    base = soma_emails_dict.get(classe)

    # Quantidade total e distinta de palavras nessa classe no treino
    tamanho_vocabulario = base.shape[0]
    total_termos_classe = sum(base) 

    produtorio = 1.0
    for idx, contagem in enumerate(vetor):
        if contagem != 0:
            prob_ter_a_palavra = calcular_frequencia_palavra(base[idx], total_termos_classe, tamanho_vocabulario)
            
            # Se a palavra aparecer mais de uma vez no novo e-mail, elevamos a probabilidade pela quantidade de ocorrências
            produtorio *= (prob_ter_a_palavra ** contagem)

    return produtorio

In [55]:
def NaiveBayes(vetor_palavra):
    resultados_classes = {}

    for classe in labels:
        # O denominador do Teorema de Naive Bayes será o mesmo nos dois cálculos (para Spam e não Spam)
        # Como o queremos apenas comparar se a probabilidade de ser Spam é maior que a de ser Não Spam, podemos eliminar o denominador.

        classe_id = (1 if classe == 'spam' else 0)

        num = py(classe_id) * p_cond(vetor_palavra, classe_id)
        resultados_classes[classe] = num

    # Tomada de decisão
    veredito = max(resultados_classes, key=resultados_classes.get)

    return veredito

In [65]:
novo_email = ['Wanna meeting for a coffee today?']

texto_vectorizado = vectorizer.transform(novo_email).toarray()[0]
# print(texto_vectorizado)

classe_predicao = NaiveBayes(texto_vectorizado)
print(f"O e-mail foi classificado como: {classe_predicao}")

O e-mail foi classificado como: n_spam
